# Replicate baseline model -  CNN-based deep canonical correlation analysis autoencoder (CNN-DCCAE) 

In [1]:
# import libraries
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn import metrics
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from braindecode.datautil.serialization import load_concat_dataset
from braindecode.datasets import BaseConcatDataset, BaseDataset
from braindecode.preprocessing import create_fixed_length_windows
from collections import Counter
from tqdm import tqdm
import plotly.express as px
import pandas as pd

In [2]:
# Load the rescaled data
OUT_PATH = r'D:\Ana_Maria\cleaned_TUH_scaled'
tuh_preproc = load_concat_dataset(OUT_PATH, preload=False)


In [3]:
# Count how many segment we have in each class
t = []
for ds in tuh_preproc.datasets:
    ds.target_name = int(ds.description['target'])
    ds.target_name = 'target'
    t.append(int(ds.description['target']))

Counter(t)

#epileptic: 1361, non-epileptic: 258

Counter({1: 1361, 0: 258})

In [4]:
# Train, Validation and Test subject-wise split (Make sure that a subject only appears in one split)

def get_subject_id(ds):
    subj = ds.description.get("subject", None)

    # Get id from dict with his_id
    if isinstance(subj, dict) and "his_id" in subj:
        return subj["his_id"]

    # Get id from path
    path = ds.description.get("path", "")
    if isinstance(path, str):
        parts = path.split("/")
        return parts[2]

subjects = [get_subject_id(ds) for ds in tuh_preproc.datasets]
unique_subjects = sorted(set(subjects))

# Train(60%), Val(20%) and test(20%) split 
train_subj, temp_subj = train_test_split(unique_subjects, test_size=0.4, random_state=36)
val_subj, test_subj = train_test_split(temp_subj, test_size=0.5, random_state=36)

train_subj, val_subj, test_subj = set(train_subj), set(val_subj), set(test_subj)

# Create Datasets
train_set = BaseConcatDataset([ds for ds in tuh_preproc.datasets if get_subject_id(ds) in train_subj])
val_set   = BaseConcatDataset([ds for ds in tuh_preproc.datasets if get_subject_id(ds) in val_subj])
test_set  = BaseConcatDataset([ds for ds in tuh_preproc.datasets if get_subject_id(ds) in test_subj])


In [5]:
# Generate compute windows
sfreq = 256 # Hz
window_size_samples = int(5 * sfreq) # 5 sec windows
window_stride_samples = int(2.5 * sfreq) # 50% overlap

def create_windows(dataset):
    return create_fixed_length_windows(
        dataset,
        window_size_samples=window_size_samples,
        window_stride_samples=window_stride_samples,
        drop_last_window=True,
        n_jobs=1,
    )

In [6]:
# Create a Dataset with separated eeg and ecg signals 

class PairedEEGECGDataset(Dataset):
    def __init__(self, windows_dataset, eeg_indices, ecg_indices):

        self.dataset = windows_dataset
        self.eeg_idx = eeg_indices
        self.ecg_idx = ecg_indices

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        x, y, meta = self.dataset[idx]

        # convert to tensor
        x = torch.as_tensor(x, dtype=torch.float32)

        eeg = x[self.eeg_idx]
        ecg = x[self.ecg_idx]
        
        # Normalise
        eeg = (eeg - eeg.mean(dim=-1, keepdim=True)) / (eeg.std(dim=-1, keepdim=True) + 1e-8)
        ecg = (ecg - ecg.mean(dim=-1, keepdim=True)) / (ecg.std(dim=-1, keepdim=True) + 1e-8)
        
        return eeg, ecg, y, meta

In [7]:
ecg_ch = ['EKG']
eeg_ch = [ 'C3', 'C4', 'CZ', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'O1', 'O2', 'P3', 'P4', 'PZ', 'T3', 'T4', 'T5', 'T6']

# Window all 20 channels at once
train_windows = create_windows(train_set)
val_windows   = create_windows(val_set)
test_windows  = create_windows(test_set)
print('Windows created')

# Get channel indices
ch_names    = train_windows.datasets[0].raw.ch_names
eeg_indices = [ch_names.index(ch) for ch in eeg_ch]
ecg_indices = [ch_names.index(ch) for ch in ecg_ch]
print("EEG indices:", eeg_indices)
print("ECG indices:", ecg_indices)

# Separate EEG from ECG signals in one Dataset
train_paired = PairedEEGECGDataset(train_windows, eeg_indices, ecg_indices)
val_paired   = PairedEEGECGDataset(val_windows,   eeg_indices, ecg_indices)
test_paired  = PairedEEGECGDataset(test_windows,  eeg_indices, ecg_indices)
print('Paired Datasets created')

# DataLoaders
train_loader = DataLoader(train_paired, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_paired,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_paired,  batch_size=32, shuffle=False)
print('Data Loaders created!')

Windows created
EEG indices: [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
ECG indices: [3]
Paired Datasets created
Data Loaders created!


In [8]:
# # Balance Classes (for classification phase)

# labels = [train_paired[i][2] for i in range(len(train_paired))]
# class_counts = np.bincount(labels)
# sample_weights = [1.0 / class_counts[l] for l in labels]
# sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
# print('Sampler created')

# train_loader_balanced = DataLoader(train_paired, batch_size=32, sampler=sampler)

Sampler was repeating minority class windows, the model was overfitting and causing a huge gap between train and validation. But since the DCCAE reconstrution loss does not depend on the labels we can pretrain the model with the class imbalance and then intoduce the sampler for classifier fine-tuning. 

In [9]:
# CCA Loss taken and adapt from https://github.com/itsikad/cca-loss-TF/blob/master/src/cca_loss.py

def matrix_sqrt_inv(A):
    """
    Computes A^(-1/2) via eigendecomposition.
    """

    A = (A + A.T) / 2
    eigvals, eigvecs = torch.linalg.eigh(A)  
    

    eigvals = eigvals.clamp(min=1e-8)
    
    sqrt_inv = eigvecs @ torch.diag(1.0 / torch.sqrt(eigvals)) @ eigvecs.T
    return sqrt_inv


class CCA_Loss(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x1, x2, r1=1e-4, r2=1e-4):
        N = x1.shape[0]
        N_float = float(N)
        scale_factor = 1.0 / (N_float - 1.0)
        device = x1.device
        dtype = x1.dtype

        scale_mat = torch.eye(N, device=device, dtype=dtype) - (1.0 / N_float) * torch.ones((N, N), device=device, dtype=dtype)

        h1_bar = x1.T @ scale_mat
        h2_bar = x2.T @ scale_mat

        cov_11 = scale_factor * (h1_bar @ h1_bar.T) + r1 * torch.eye(x1.shape[1], device=device, dtype=dtype)
        cov_22 = scale_factor * (h2_bar @ h2_bar.T) + r2 * torch.eye(x2.shape[1], device=device, dtype=dtype)
        cov_12 = scale_factor * (h1_bar @ h2_bar.T)

        
        cov_11_sqrt_inv = matrix_sqrt_inv(cov_11)
        cov_22_sqrt_inv = matrix_sqrt_inv(cov_22)

        R = cov_11_sqrt_inv @ cov_12 @ cov_22_sqrt_inv

        U, s, Vh = torch.linalg.svd(R, full_matrices=False)
        V = Vh.T

        loss = -torch.sum(s)

        ctx.save_for_backward(
            h1_bar, h2_bar,
            cov_11_sqrt_inv, cov_22_sqrt_inv,
            U, s, V,
            torch.tensor(scale_factor, device=device, dtype=dtype)
        )

        return loss

    @staticmethod
    def backward(ctx, grad_output):
        h1_bar, h2_bar, cov_11_sqrt_inv, cov_22_sqrt_inv, U, s, V, scale_factor = ctx.saved_tensors

        cov_11_sqrt_inv_u = cov_11_sqrt_inv @ U
        cov_22_sqrt_inv_v = cov_22_sqrt_inv @ V

        S = torch.diag(s)

        delta_11 = -0.5 * cov_11_sqrt_inv_u @ S @ cov_11_sqrt_inv_u.T
        delta_22 = -0.5 * cov_22_sqrt_inv_v @ S @ cov_22_sqrt_inv_v.T
        delta_12 = cov_11_sqrt_inv @ U @ V.T @ cov_22_sqrt_inv

        grad_h1 = -scale_factor * (2.0 * delta_11 @ h1_bar + delta_12 @ h2_bar)
        grad_h2 = -scale_factor * (2.0 * delta_22 @ h2_bar + delta_12.T @ h1_bar)

        grad_x1 = grad_h1.T * grad_output
        grad_x2 = grad_h2.T * grad_output

        return grad_x1, grad_x2, None, None


def cca_loss(x1, x2, r1=1e-4, r2=1e-4):
    return CCA_Loss.apply(x1, x2, r1, r2)

First I replicated the model of the paper:

    - Encoder: 4 conv blocks with MaxPool, no activation (linear);
    - Decoder: transposed convolutions;
    - CCA loss;

The model was overfitting and not learning anything.

Fixes:

    - CCA loss was negative so I've changed the loss function to: total_loss = eeg_loss + ecg_loss - lambda_r * abs(correlation)

    - Increased lambda_r from 1e-10 (paper value) to 1e-3 to make CCA contribution more meaningful

    - Added LeakyRelu activation to prevent dead neurons while enabling nonlinear learning (Paper only uses linear activations bu it was causing near-identity transformations and the model stuck at loss ~1.83; Then I tried ReLU but the ECG decoder was collapsing to zero)

    - added a different and simpler decoder for the ECG for a single-channel signal reconstruction. (the original decoder was shared with EEG and ECG and the ECG decoder was collapsing)

    - Added BatchNorm after each conv block to stabilise training

    - Added dropout for regularization so it could prevent overfitting.

    - For the optimizer I started using the Adam with the same learning rate and a global weight decay, as in the paper, but it was collapsing the EEG encoder weights to zero. So I'm only adding weight decay to the decoder to improve the reconstruction, but not on the encoder, in order to preserve the learned representations.

    - I'm starting with a lower learning rate, trying to prevent that the model gets stuck in a local minimum (or that the model overshoot optimal point in epoch 1) and added the schedule to change the learing rate while training and also early stopping

    - Increased the number of filters to 32/64 filters (Paper uses 10 filters which was stucking the loss at ~1.87 and it was predicting the signal mean), with this change the loss dropped to ~1.66.




In [10]:
# Set the number of channels
n_channels = 19
num_samples = 5 * 256 # 5 sec with a fs of 256 Hz
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class CNNEncoder(nn.Module):
    def __init__(self, n_channels, T=num_samples, dropout=0.3):
        super().__init__()
        self.encoder_block = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=5, padding=2),  
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.BatchNorm1d(32, momentum=0.01),
            nn.Dropout(dropout),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),          
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(5),
            nn.BatchNorm1d(64, momentum=0.01),
            nn.Dropout(dropout),

            nn.Conv1d(64, 32, kernel_size=5, padding=2),          
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(5),
            nn.BatchNorm1d(32, momentum=0.01),
            nn.Dropout(dropout),

            nn.Conv1d(32, 1, kernel_size=1),
            nn.Flatten(),
            nn.BatchNorm1d(T // (2 * 5 * 5), momentum=0.01)
        )
        # latent dim = T // (2*5*5) = 25
    def forward(self, x):
        return self.encoder_block(x)
        


class CNNDecoder(nn.Module):
    def __init__(self, out_channels, T=num_samples, dropout=0.3):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(32, 64, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Conv1d(64, 32, kernel_size=5, padding=2),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(32, 16, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Conv1d(16, 8, kernel_size=5, padding=2),
            nn.LeakyReLU(0.1),
            nn.ConvTranspose1d(8, 8, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Conv1d(8, 10, kernel_size=1),
            nn.Upsample(size=T),
            nn.Conv1d(10, out_channels, kernel_size=1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)   
        return self.convs(x)
    
class ECGDecoder(nn.Module):
    def __init__(self, T=num_samples, dropout=0.3):
        super().__init__()
        self.convs = nn.Sequential(
            nn.ConvTranspose1d(1, 32, kernel_size=5, stride=2, padding=2, output_padding=1),  
            nn.BatchNorm1d(32, momentum=0.01),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.ConvTranspose1d(32, 64, kernel_size=5, stride=5, padding=2, output_padding=4), 
            nn.BatchNorm1d(64, momentum=0.01),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=5, padding=2, output_padding=4),
            nn.BatchNorm1d(32, momentum=0.01),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),

            nn.Upsample(size=T),                  
            nn.Conv1d(32, 1, kernel_size=1)         
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        return self.convs(x)

In [ ]:
class DCCAE(nn.Module):

    def __init__(self, n_eeg_channels = 19, T = num_samples, lambda_r = 1e-10):
        super().__init__()
        self.lambda_r = lambda_r

        self.eeg_encoder = CNNEncoder(n_channels=n_eeg_channels, T=T)
        self.ecg_encoder = CNNEncoder(n_channels=1, T=T)
        self.eeg_decoder = CNNDecoder(out_channels=n_eeg_channels, T=T)
        self.ecg_decoder = ECGDecoder(T=T)

    def encode(self, eeg, ecg):
        return self.eeg_encoder(eeg), self.ecg_encoder(ecg)
    
    def forward(self, eeg, ecg):
        z_eeg = self.eeg_encoder(eeg)
        z_ecg = self.ecg_encoder(ecg)
        r_eeg = self.eeg_decoder(z_eeg)
        r_ecg = self.ecg_decoder(z_ecg)

        return z_eeg, z_ecg, r_eeg, r_ecg
    
    # Loss function
    def loss(self, eeg, ecg, lambda_r = 1e-3):
        """
        lambda_r = 1e-10 to tune the model and then increased to 1e-3 so that the correlation has more weight
        DCCAE loss = MSE(EEG) + MSE(ECG) - lambda * CCA_loss
        """
        z_eeg, z_ecg, r_eeg, r_ecg = self.forward(eeg, ecg)

        # Reconstruction losses using Mean Squared Error 
        eeg_loss = F.mse_loss(r_eeg, eeg)
        ecg_loss = F.mse_loss(r_ecg, ecg)

        # Correlation loss
        correlation = cca_loss(z_eeg, z_ecg)
        #correlation = cossine_similarity_loss(z_eeg, z_ecg)

        # Loss function (I'm using the absolute value of the correlation because I checked that the cca loss is negative and since we want to maximize it it needed to be positive in this formula)
        total_loss = eeg_loss + ecg_loss - lambda_r * abs(correlation)

        return total_loss, {
            "total_loss": total_loss.item(),
            "eeg_loss": eeg_loss.item(),
            "ecg_loss": ecg_loss.item(),
            "correlation_loss": correlation.item(),
        }

    
model = DCCAE().to(device)
print(model)


DCCAE(
  (eeg_encoder): CNNEncoder(
    (encoder_block): Sequential(
      (0): Conv1d(19, 32, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): LeakyReLU(negative_slope=0.1)
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): BatchNorm1d(32, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
      (4): Dropout(p=0.3, inplace=False)
      (5): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (6): LeakyReLU(negative_slope=0.1)
      (7): MaxPool1d(kernel_size=5, stride=5, padding=0, dilation=1, ceil_mode=False)
      (8): BatchNorm1d(64, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
      (9): Dropout(p=0.3, inplace=False)
      (10): Conv1d(64, 32, kernel_size=(5,), stride=(1,), padding=(2,))
      (11): LeakyReLU(negative_slope=0.1)
      (12): MaxPool1d(kernel_size=5, stride=5, padding=0, dilation=1, ceil_mode=False)
      (13): BatchNorm1d(32, eps=1e-05, momentum=0.01, affine=True, t

In [ ]:
# Sanity check
model.train()
eeg, ecg, y, _ = next(iter(train_loader))
eeg, ecg = eeg.to(device), ecg.to(device)

print("EEG input shape:", eeg.shape)   
print("ECG input shape:", ecg.shape)   

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
optimizer.zero_grad()
loss, loss_dict = model.loss(eeg, ecg)

print("Loss:", loss_dict)
print("Total loss:", loss.item())

loss.backward()

total_grad = sum(p.grad.abs().sum().item() for p in model.parameters() if p.grad is not None)
print("Total gradient magnitude:", total_grad)

# Check model and reconstructions 
with torch.no_grad():
    z_eeg, z_ecg, r_eeg, r_ecg = model(eeg, ecg)
    print("z_eeg shape:", z_eeg.shape)  
    print("z_ecg shape:", z_ecg.shape)  
    print("r_eeg shape:", r_eeg.shape)  
    print("r_ecg shape:", r_ecg.shape)   
    print("Input EEG mean/std:", eeg.mean().item(), eeg.std().item())
    print("Recon EEG mean/std:", r_eeg.mean().item(), r_eeg.std().item())
    print("Input ECG mean/std:", ecg.mean().item(), ecg.std().item())
    print("Recon ECG mean/std:", r_ecg.mean().item(), r_ecg.std().item())

EEG input shape: torch.Size([32, 19, 1280])
ECG input shape: torch.Size([32, 1, 1280])

Loss components: {'total_loss': 2.245851993560791, 'eeg_loss': 1.0582705736160278, 'ecg_loss': 1.2087029218673706, 'correlation_loss': -21.12152099609375}
Total loss: 2.245851993560791
Total gradient magnitude: 121.52042108287787

z_eeg shape: torch.Size([32, 25])
z_ecg shape: torch.Size([32, 25])
r_eeg shape: torch.Size([32, 19, 1280])
r_ecg shape: torch.Size([32, 1, 1280])

Input EEG mean/std: -1.9606791190618367e-10 0.9996099472045898
Recon EEG mean/std: 0.020056281238794327 0.24213673174381256
Input ECG mean/std: 0.0 0.9996214509010315
Recon ECG mean/std: 0.10689113289117813 0.4611285328865051


In [ ]:
model = DCCAE().to(device)

optimizer = torch.optim.Adam([
    {'params': model.eeg_encoder.parameters(), 'weight_decay': 0.0},
    {'params': model.ecg_encoder.parameters(), 'weight_decay': 0.0}, # No weight decay on the encoders
    {'params': model.eeg_decoder.parameters(), 'weight_decay': 1e-4}, # Added only on the decoders
    {'params': model.ecg_decoder.parameters(), 'weight_decay': 1e-4},
], lr=3e-4) # start from reduced lr, before was lr=1e-3

# Scheduler to lower the learning rate if the model do not improve
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

best_val_loss = float('inf')
epochs = 30
patience = 10
epochs_no_improve = 0

c:\Users\student\anaconda3\envs\dl_ana\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
train_losses = []
val_losses = []
best_val_loss = float('inf')
epochs_no_improve = 0


for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0

    for eeg, ecg, y, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False):
        eeg = eeg.to(device)
        ecg = ecg.to(device)

        optimizer.zero_grad()
        loss, loss_dict = model.loss(eeg, ecg)

        loss.backward()
        # prevent gradients from becoming too large and explode
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    

    # Validation
    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for eeg, ecg, y, _ in val_loader:
            eeg = eeg.to(device)
            ecg = ecg.to(device)
            loss, _ = model.loss(eeg, ecg)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    
    
    val_losses.append(avg_val_loss)

    scheduler.step(avg_val_loss)

    # Print current lr
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1} - Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | LR: {current_lr:.2e}')
    

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_dccae_lr.pth')
        print(f'New best model saved (val loss: {best_val_loss:.4f})')
        epochs_no_improve = 0
    
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()
plt.show()

KeyboardInterrupt: 

# Evaluation

In [16]:
# Quick check on a batch
model.load_state_dict(torch.load('best_dccae_lr.pth', weights_only=True))
model.eval()

with torch.no_grad():
    eeg, ecg, y, _ = next(iter(val_loader))
    eeg, ecg = eeg.to(device), ecg.to(device)
    loss, loss_dict = model.loss(eeg, ecg)
    print(loss_dict)

{'total_loss': 1.8470420837402344, 'eeg_loss': 0.9606649875640869, 'ecg_loss': 0.9072862863540649, 'correlation_loss': -20.909231185913086}


In [ ]:
# Encode test set
model.eval()
all_z_eeg, all_z_ecg, all_y = [], [], []

with torch.no_grad():
    for eeg, ecg, y, _ in tqdm(test_loader, desc='Encoding test set'):
        eeg, ecg = eeg.to(device), ecg.to(device)
        z_eeg, z_ecg = model.encode(eeg, ecg)
        all_z_eeg.append(z_eeg.cpu())
        all_z_ecg.append(z_ecg.cpu())
        all_y.append(y)

all_z_eeg = torch.cat(all_z_eeg).numpy()
all_z_ecg = torch.cat(all_z_ecg).numpy()
all_y = torch.cat(all_y).numpy()

# Concat EEG and ECG representations
z_fused = np.concatenate([all_z_eeg, all_z_ecg], axis=1)

Encoding test set: 100%|██████████| 6627/6627 [08:01<00:00, 13.76it/s]


For the cluster evaluation I'm testing first with KMeans for a kick evaluations (because its faster) and then I test it with SpectralClustering as in the paper (it takes a long time to run on large datasets).

The evaluation metrics are then the Clustering accuracy and the RIOC (to compare with the paper)

In [18]:
from sklearn.cluster import MiniBatchKMeans

# Check with KMeans first (faster but less acurate)
kmeans = MiniBatchKMeans(n_clusters=2, random_state=42)
cluster_labels = kmeans.fit_predict(z_fused)

c:\Users\student\anaconda3\envs\dl_ana\lib\site-packages\sklearn\cluster\_kmeans.py:1955: UserWarning: MiniBatchKMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can prevent it by setting batch_size >= 2048 or by setting the environment variable OMP_NUM_THREADS=4
  warnings.warn(


In [ ]:
# In the paper they evaluated with SpectralClustering (it takes longer to run, I still didn't try it)
from sklearn.cluster import SpectralClustering


clustering = SpectralClustering(
    n_clusters=2,
    affinity='nearest_neighbors',
    n_neighbors=10,
    random_state=42,
    n_jobs=-1
)
cluster_labels = clustering.fit_predict(z_fused)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy.optimize import linear_sum_assignment

def clustering_accuracy(y_true, y_pred):
    """Match cluster labels to true labels using Hungarian algorithm."""
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm) # Hungarian matching to find the best mapping
    matched = np.zeros_like(y_pred)
    for r, c in zip(row_ind, col_ind):
        matched[y_pred == c] = r
    return accuracy_score(y_true, matched), matched

acc, matched_labels = clustering_accuracy(all_y, cluster_labels)
rioc = (acc - 0.5) / 0.5 * 100 # RIOC measures improvement over chance.

print(f'Clustering Accuracy: {acc*100:.3f}%  (Paper: 68.704%)')
print(f'RIOC: {rioc:.2f}%  (Paper: 37.41%)')
print(classification_report(all_y, matched_labels, target_names=['Non-epileptic', 'Epileptic']))

Clustering Accuracy: 61.575%  (Paper: 68.704%)
RIOC: 23.15%  (Paper: 37.41%)
               precision    recall  f1-score   support

Non-epileptic       0.10      0.37      0.16     20748
    Epileptic       0.90      0.64      0.75    191299

     accuracy                           0.62    212047
    macro avg       0.50      0.50      0.45    212047
 weighted avg       0.82      0.62      0.69    212047



To be more precise I'm then evaluating on a subject level using majority voting

In [ ]:
# Get subject ID for each test window
test_subjects = []
for ds in test_windows.datasets:
    subj = get_subject_id(ds)
    n_windows = len(ds)  
    test_subjects.extend([subj] * n_windows)

test_subjects = np.array(test_subjects)


# Majority vote per subject

# Collect predictions and true label
subject_preds = {}
subject_true = {}

for subj, pred, true in zip(test_subjects, matched_labels, all_y):
    # Subject appear for the first time 
    if subj not in subject_preds:
        subject_preds[subj] = []
        subject_true[subj] = true
    # Add pediction to the dict
    subject_preds[subj].append(pred)

# Majority vote
subj_pred_final = [Counter(subject_preds[s]).most_common(1)[0][0] for s in subject_preds] # finds the most frequent prediction for each subject.
subj_true_final = [subject_true[s] for s in subject_preds] # True labels

subj_acc = accuracy_score(subj_true_final, subj_pred_final)
subj_rioc = (subj_acc - 0.5) / 0.5 * 100
print(f'Subject-level Accuracy: {subj_acc*100:.3f}%')
print(f'Subject-level RIOC: {subj_rioc:.2f}%')

['aaaaabhz' 'aaaaabhz' 'aaaaabhz' ... 'aaaaapsm' 'aaaaapsm' 'aaaaapsm']
Subject-level Accuracy: 42.500%
Subject-level RIOC:     -15.00%


# Cluster visualization

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
z = reducer.fit_transform(z_fused)

df = pd.DataFrame({
    'x': z[:, 0],
    'y': z[:, 1],
    'label': ['Epileptic' if y == 1 else 'Non-epileptic' for y in all_y],
    'cluster': [f'Cluster {c}' for c in cluster_labels]
})

# Plot by true label
fig1 = px.scatter(df, x='x', y='y', color='label',
                  color_discrete_map={'Epileptic': 'tomato', 'Non-epileptic': 'steelblue'},
                  opacity=0.5, title='UMAP — True Labels (Test Set)')
fig1.update_traces(marker=dict(size=3))
fig1.show()

# Plot by cluster assignment
fig2 = px.scatter(df, x='x', y='y', color='cluster',
                  opacity=0.5, title='UMAP — Spectral Clustering (Test Set)')
fig2.update_traces(marker=dict(size=3))
fig2.show()

df["subject"] = test_subjects

fig = px.scatter(
    df,
    x="x",
    y="y",
    color="subject"
)

c:\Users\student\anaconda3\envs\dl_ana\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


By this representaions we can see that the classes are very mixed, so the encoder has not learned.

By the spectral clustering, it has found something real, but not related to the separation between epileptic and non-epileptic.

(i'm going to try to plot by subject id to check if that is the bias)

I still did not introduce the sampler for balancing the classes tho